# TheraBot Model Evaluation for On-Device Deployment

This notebook evaluates three models for mobile/edge deployment:
- **llama3.2-1b-instruct**: Base model (fp32)
- **therabot-fp32**: Fine-tuned model (fp32)
- **therabot-int8.gguf**: Quantized model (int8)

## Evaluation Metrics
- 🚀 **Performance**: Latency, tokens/second
- 💾 **Memory**: Peak usage, efficiency
- 🎯 **Quality**: BLEU score, perplexity
- 🔋 **Power**: Estimated consumption
- 📦 **Size**: Model size, disk usage

## 1. Setup Environment

In [ ]:
# Install required packages
!pip install -q mlflow
!pip install -q transformers datasets accelerate evaluate
!pip install -q psutil memory-profiler GPUtil
!pip install -q llama-cpp-python
!pip install -q bitsandbytes

In [ ]:
# Mount Google Drive to access models (if using Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload evaluation.py file
from google.colab import files
uploaded = files.upload()
# Make sure evaluation.py is uploaded

In [ ]:
# Import the evaluation framework
from evaluation import ModelEvaluator, ModelConfig, BenchmarkResults
import mlflow
import torch
import os

## 2. Configure Models

Update the paths below to match your model locations.

In [ ]:
# Model configurations
model_configs = [
    ModelConfig(
        name="llama3.2-1b-base",
        path="meta-llama/Llama-3.2-1B-Instruct",
        model_type="transformers",
        precision="fp32"
    ),
    ModelConfig(
        name="therabot-fp32",
        path="/content/outputs-llama1b-counselor",  # Update this path
        model_type="transformers",
        precision="fp32"
    ),
    ModelConfig(
        name="therabot-int8",
        path="/content/therabot_int8.gguf",  # Update this path
        model_type="gguf",
        precision="int8"
    )
]

print("Configured models:")
for config in model_configs:
    print(f"  - {config.name} ({config.model_type}, {config.precision})")

## 3. Initialize Evaluator

In [ ]:
# Initialize the evaluator
evaluator = ModelEvaluator(experiment_name="therabot_ondevice_comparison")

# Load test dataset
print("Loading test dataset...")
evaluator.load_test_dataset(num_samples=50)  # Use smaller sample for faster evaluation

print("\nEvaluator initialized successfully!")

## 4. Run Individual Model Evaluations

Evaluate each model separately to see detailed metrics.

### 4.1 Evaluate Base Llama 3.2-1B Model

In [ ]:
# Evaluate base model
base_model_config = model_configs[0]  # llama3.2-1b-base
print(f"Evaluating: {base_model_config.name}")

base_results = evaluator.run_comprehensive_evaluation(base_model_config)

if base_results:
    print(f"\n✅ {base_model_config.name} evaluation complete!")
    print(f"   Avg Latency: {base_results.avg_latency_ms:.1f}ms")
    print(f"   Tokens/sec: {base_results.tokens_per_second:.1f}")
    print(f"   Peak Memory: {base_results.peak_memory_mb:.1f}MB")
    print(f"   Model Size: {base_results.model_size_mb:.1f}MB")
else:
    print(f"❌ Failed to evaluate {base_model_config.name}")

### 4.2 Evaluate Fine-tuned TheraBot (FP32)

In [ ]:
# Evaluate fine-tuned fp32 model
fp32_model_config = model_configs[1]  # therabot-fp32
print(f"Evaluating: {fp32_model_config.name}")

fp32_results = evaluator.run_comprehensive_evaluation(fp32_model_config)

if fp32_results:
    print(f"\n✅ {fp32_model_config.name} evaluation complete!")
    print(f"   Avg Latency: {fp32_results.avg_latency_ms:.1f}ms")
    print(f"   Tokens/sec: {fp32_results.tokens_per_second:.1f}")
    print(f"   Peak Memory: {fp32_results.peak_memory_mb:.1f}MB")
    print(f"   Model Size: {fp32_results.model_size_mb:.1f}MB")
else:
    print(f"❌ Failed to evaluate {fp32_model_config.name}")

### 4.3 Evaluate Quantized TheraBot (INT8)

In [ ]:
# Evaluate quantized int8 model
int8_model_config = model_configs[2]  # therabot-int8
print(f"Evaluating: {int8_model_config.name}")

int8_results = evaluator.run_comprehensive_evaluation(int8_model_config)

if int8_results:
    print(f"\n✅ {int8_model_config.name} evaluation complete!")
    print(f"   Avg Latency: {int8_results.avg_latency_ms:.1f}ms")
    print(f"   Tokens/sec: {int8_results.tokens_per_second:.1f}")
    print(f"   Peak Memory: {int8_results.peak_memory_mb:.1f}MB")
    print(f"   Model Size: {int8_results.model_size_mb:.1f}MB")
else:
    print(f"❌ Failed to evaluate {int8_model_config.name}")

## 5. Comprehensive Model Comparison

Run all models and generate a detailed comparison report.

In [ ]:
# Run comprehensive comparison
print("Starting comprehensive model comparison...")
print("This may take 10-15 minutes depending on your hardware.\n")

all_results = evaluator.compare_models(model_configs)

print("\n🎉 Evaluation completed successfully!")
print(f"Evaluated {len(all_results)} models.")

## 6. Detailed Analysis and Visualizations

In [ ]:
# Create visualizations for comparison
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Extract data for visualization
model_names = list(all_results.keys())
latencies = [all_results[name].avg_latency_ms for name in model_names]
memory_usage = [all_results[name].peak_memory_mb for name in model_names]
model_sizes = [all_results[name].model_size_mb for name in model_names]
tokens_per_sec = [all_results[name].tokens_per_second for name in model_names]
bleu_scores = [all_results[name].bleu_score for name in model_names]
power_consumption = [all_results[name].estimated_power_mw for name in model_names]

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': model_names,
    'Latency (ms)': latencies,
    'Memory (MB)': memory_usage,
    'Size (MB)': model_sizes,
    'Tokens/sec': tokens_per_sec,
    'BLEU Score': bleu_scores,
    'Power (mW)': power_consumption
})

print("Model Comparison Summary:")
print(comparison_df.round(2))

In [ ]:
# Performance vs Quality Trade-off Plot
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Latency comparison
ax1.bar(model_names, latencies, color=['skyblue', 'lightgreen', 'orange'])
ax1.set_title('Average Latency Comparison')
ax1.set_ylabel('Latency (ms)')
ax1.tick_params(axis='x', rotation=45)

# Memory usage comparison
ax2.bar(model_names, memory_usage, color=['skyblue', 'lightgreen', 'orange'])
ax2.set_title('Peak Memory Usage')
ax2.set_ylabel('Memory (MB)')
ax2.tick_params(axis='x', rotation=45)

# Model size comparison
ax3.bar(model_names, model_sizes, color=['skyblue', 'lightgreen', 'orange'])
ax3.set_title('Model Size')
ax3.set_ylabel('Size (MB)')
ax3.tick_params(axis='x', rotation=45)

# Performance vs Quality scatter
ax4.scatter(latencies, bleu_scores, s=[size/10 for size in model_sizes], 
           c=['skyblue', 'lightgreen', 'orange'], alpha=0.7)
for i, name in enumerate(model_names):
    ax4.annotate(name, (latencies[i], bleu_scores[i]), 
                xytext=(5, 5), textcoords='offset points')
ax4.set_xlabel('Latency (ms)')
ax4.set_ylabel('BLEU Score')
ax4.set_title('Performance vs Quality Trade-off')

plt.tight_layout()
plt.show()

In [ ]:
# On-device deployment radar chart
import math

# Normalize metrics for radar chart (0-1 scale, higher is better)
def normalize_metric(values, reverse=False):
    if reverse:  # For metrics where lower is better (latency, memory, size, power)
        return [(max(values) - v) / (max(values) - min(values)) for v in values]
    else:  # For metrics where higher is better (tokens/sec, BLEU)
        return [(v - min(values)) / (max(values) - min(values)) for v in values]

# Prepare data
metrics = ['Speed', 'Memory Eff.', 'Size Eff.', 'Quality', 'Tokens/sec', 'Power Eff.']
normalized_data = {
    name: [
        normalize_metric(latencies, reverse=True)[i],  # Speed (lower latency = better)
        normalize_metric(memory_usage, reverse=True)[i],  # Memory efficiency
        normalize_metric(model_sizes, reverse=True)[i],  # Size efficiency
        normalize_metric(bleu_scores)[i],  # Quality
        normalize_metric(tokens_per_sec)[i],  # Tokens per second
        normalize_metric(power_consumption, reverse=True)[i]  # Power efficiency
    ] for i, name in enumerate(model_names)
}

# Create radar chart
angles = [n / float(len(metrics)) * 2 * math.pi for n in range(len(metrics))]
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = ['skyblue', 'lightgreen', 'orange']
for i, (name, values) in enumerate(normalized_data.items()):
    values += values[:1]  # Complete the circle
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=colors[i])
    ax.fill(angles, values, alpha=0.25, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_title('On-Device Deployment Characteristics\n(Larger area = better for mobile)', 
             size=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True)

plt.show()

## 7. Mobile Deployment Recommendations

In [ ]:
# Generate deployment recommendations
print("MOBILE DEPLOYMENT ANALYSIS")
print("=" * 50)

# Calculate composite scores for different use cases
def calculate_composite_score(results, weights):
    """Calculate weighted composite score"""
    scores = {}
    
    # Normalize all metrics to 0-1 scale
    all_latencies = [r.avg_latency_ms for r in results.values()]
    all_memory = [r.peak_memory_mb for r in results.values()]
    all_sizes = [r.model_size_mb for r in results.values()]
    all_bleu = [r.bleu_score for r in results.values()]
    all_power = [r.estimated_power_mw for r in results.values()]
    
    for name, result in results.items():
        # Normalize metrics (higher score = better)
        speed_score = (max(all_latencies) - result.avg_latency_ms) / (max(all_latencies) - min(all_latencies))
        memory_score = (max(all_memory) - result.peak_memory_mb) / (max(all_memory) - min(all_memory))
        size_score = (max(all_sizes) - result.model_size_mb) / (max(all_sizes) - min(all_sizes))
        quality_score = (result.bleu_score - min(all_bleu)) / (max(all_bleu) - min(all_bleu))
        power_score = (max(all_power) - result.estimated_power_mw) / (max(all_power) - min(all_power))
        
        # Calculate weighted score
        composite = (
            weights['speed'] * speed_score +
            weights['memory'] * memory_score +
            weights['size'] * size_score +
            weights['quality'] * quality_score +
            weights['power'] * power_score
        )
        scores[name] = composite
    
    return scores

# Different use case scenarios
scenarios = {
    'Real-time Chat': {'speed': 0.4, 'memory': 0.2, 'size': 0.1, 'quality': 0.2, 'power': 0.1},
    'Battery Constrained': {'speed': 0.1, 'memory': 0.2, 'size': 0.2, 'quality': 0.2, 'power': 0.3},
    'High Quality': {'speed': 0.1, 'memory': 0.1, 'size': 0.1, 'quality': 0.6, 'power': 0.1},
    'Resource Limited': {'speed': 0.2, 'memory': 0.3, 'size': 0.3, 'quality': 0.1, 'power': 0.1},
    'Balanced': {'speed': 0.2, 'memory': 0.2, 'size': 0.2, 'quality': 0.2, 'power': 0.2}
}

for scenario_name, weights in scenarios.items():
    scores = calculate_composite_score(all_results, weights)
    best_model = max(scores.items(), key=lambda x: x[1])
    print(f"\n🎯 {scenario_name}:")
    print(f"   Best Choice: {best_model[0]} (Score: {best_model[1]:.3f})")
    
    # Show all scores for this scenario
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    for i, (model, score) in enumerate(sorted_scores):
        rank = "🥇" if i == 0 else "🥈" if i == 1 else "🥉"
        print(f"   {rank} {model}: {score:.3f}")

## 8. MLflow Results

View all results in MLflow UI for detailed tracking and comparison.

In [ ]:
# Display MLflow tracking info
print("🔬 MLflow Experiment Tracking")
print("=" * 40)
print(f"Experiment: {evaluator.experiment_name}")
print(f"Total runs: {len(all_results)}")
print("\nTo view detailed results:")
print("1. Run: mlflow ui --port 5000")
print("2. Open: http://localhost:5000")
print("3. Navigate to your experiment")

# Save results summary
summary_file = "model_comparison_summary.json"
summary_data = {
    'experiment_name': evaluator.experiment_name,
    'models_evaluated': len(all_results),
    'results': {
        name: {
            'avg_latency_ms': result.avg_latency_ms,
            'tokens_per_second': result.tokens_per_second,
            'peak_memory_mb': result.peak_memory_mb,
            'model_size_mb': result.model_size_mb,
            'bleu_score': result.bleu_score,
            'estimated_power_mw': result.estimated_power_mw
        } for name, result in all_results.items()
    }
}

import json
with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Results saved to: {summary_file}")

# Download summary file
try:
    from google.colab import files
    files.download(summary_file)
    print("📥 Summary file downloaded!")
except:
    print("📁 Summary file saved locally")

## 9. Conclusion and Next Steps

Based on the evaluation results, you now have comprehensive data to make informed decisions about model deployment:

### Key Insights:
- **Performance**: Which model offers the best latency for real-time applications
- **Efficiency**: Memory and power consumption trade-offs
- **Quality**: How quantization affects model accuracy
- **Size**: Storage requirements for different model variants

### Recommended Next Steps:
1. **Test on Target Hardware**: Run these benchmarks on your actual mobile devices
2. **A/B Testing**: Deploy multiple models and test with real users
3. **Further Optimization**: Consider pruning, knowledge distillation, or dynamic quantization
4. **Monitoring**: Set up production monitoring for the chosen model

### Model Selection Guidelines:
- **High-end devices**: therabot-fp32 for best quality
- **Mid-range devices**: therabot-int8 for balanced performance
- **Low-end devices**: Consider further quantization (INT4) or smaller model variants